# Probe再解析・独立評価実験

2026-11-10。11月4日のPhi-3.5-mini-instructの保存済みHidden Stateを再利用し、Probeの校正と新しい説明ファミリーへの汎化を検証します。

## 実行する分析
1. 学習ファミリー10個から5個をXラベルにする全252通りを列挙し、Probeを再学習（20概念ペア×5の重複する学習構成＝25,200 fit）。
2. 固定した正解ラベルProbeについて、評価ファミリー10個のラベルを全252通り置換する別検定。
3. X/Yそれぞれの評価ファミリーを1つずつ除く25通りの感度分析。
4. 新しく作成した40概念×5説明ファミリー×4プロンプト枠（800入力）のHidden Stateを同じモデル・同じ位置から抽出し、既存Probeの識別性能を評価。

## 注意
- 学習ラベル置換検定と評価ラベル置換検定は別の問いに答えます。混同しないでください。
- 5つの学習構成はファミリーを共有するため、独立した5回の追試ではありません。
- 新しい説明文はAI作成の候補刺激です。事実性・概念の一意性・難易度の確認は人手で行い、卒論の確定データにする前にレビューしてください。
- AUCが高くても、あらゆる文脈への汎化や「概念そのもの」の測定妥当性を証明するものではありません。
- 回答生成は行いません。入力時点のHidden Stateのみを抽出します。

Google ColabのGPUランタイム（T4以上）を推奨します。

In [ ]:
!pip -q install "transformers==4.57.6" "scikit-learn==1.6.1" accelerate sentencepiece scipy pandas numpy matplotlib requests

import os, json, hashlib, itertools, time, re, gc, requests
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits

ROOT = Path("/content/probe_recheck")
ROOT.mkdir(exist_ok=True)
BASE = "https://raw.githubusercontent.com/ryouy/GraduationThesis/main/1104_instruction_probe_20pairs/results/instruction-probe20-20261104T034343105831Z/"

EXPECTED = {
    "training.csv": "362e272f5177533c64504270e5b3a1978f7aaceaf949baa76f3ee96fd4c8c3e7",
    "design.csv": "fcccba9aa22b971a44f6db9a7a3156a82699e989b12c8325bf85ba3b7d18db4f",
    "config.json": "e9ee606f39f89e9584e1dfc19266b7ce7ed82f4d4d311f3f714fce578138d3c3",
    "prompt_index.json": "0be01a831381d612453e719675a63b8022c9afb761b8ab04a4cc0d06a8f771b6",
    "features_L31_assistant_start.npz": "660938d5960f741111b49162ccb459d33fd1796b10da7cfe2769d09b658983a3",
    "shuffled.csv": "fa493f21c7e785a44b81f15b8fcbb28c4e07b22a036e7c385b0ae3f70f44635b",
}
for name, expected in EXPECTED.items():
    path = ROOT / name
    if not path.exists():
        response = requests.get(BASE + name, timeout=180)
        response.raise_for_status()
        path.write_bytes(response.content)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == expected, f"SHA256 mismatch: {name}: {digest}"
    print("verified", name, path.stat().st_size)

train = pd.read_csv(ROOT / "training.csv")
design = pd.read_csv(ROOT / "design.csv")
cfg = json.loads((ROOT / "config.json").read_text())
prompt_index = json.loads((ROOT / "prompt_index.json").read_text())
lookup = {r["prompt"]: i for i, r in enumerate(prompt_index)}
F = np.load(ROOT / "features_L31_assistant_start.npz")["features"].astype(np.float64)
assert F.shape == (cfg["unique_n"], 3072)
assert len(train) == 1200 and len(design) == 6100
assert len(lookup) == cfg["unique_n"]
print("Loaded", F.shape, "hidden-state matrix")


In [ ]:
def family_auc(frame, score_col="score"):
    grouped = frame.groupby(["family", "cue_kind"], as_index=False)[score_col].mean()
    y = (grouped["cue_kind"] == "x").astype(int)
    return float(roc_auc_score(y, grouped[score_col]))

def exact_eval_label_permutation(family_frame, observed_auc):
    # Ten fixed evaluation families: five X and five Y. Enumerate C(10,5)=252 labels.
    ordered = family_frame.sort_values(["cue_kind", "family"]).reset_index(drop=True)
    scores = ordered["score"].to_numpy()
    perms = []
    for positive in itertools.combinations(range(10), 5):
        labels = np.zeros(10, dtype=int)
        labels[list(positive)] = 1
        perms.append(float(roc_auc_score(labels, scores)))
    perms = np.asarray(perms)
    return perms, float(np.mean(perms >= observed_auc - 1e-12))

def exact_training_label_permutations():
    all_null, summary, eval_perm_rows, family_swap_rows = [], [], [], []
    combinations_done = 0
    threadpool_limits(limits=1)

    for x, y in cfg["pairs"].items():
        for repeat in range(cfg["fit_repeats"]):
            tr = train[train["concept"].isin([x, y]) & (train["family_index"] != repeat)].copy()
            D = design[(design["pair"] == x) &
                       (design["condition"] == "neutral") &
                       (design["cue_kind"].isin(["x", "y"]))].copy()
            ti = np.array([lookup[p] for p in tr["prompt"]], dtype=int)
            di = np.array([lookup[p] for p in D["prompt"]], dtype=int)
            X = F[ti]
            scaler = StandardScaler().fit(X)
            Z = scaler.transform(X)
            labels = (tr["concept"] == x).astype(int).to_numpy()
            assert len(labels) == 50 and labels.sum() == 25
            true_clf = LogisticRegression(C=cfg["C"], solver="liblinear",
                                          max_iter=2000, random_state=cfg["seed"])
            true_clf.fit(Z, labels)
            actual_scores = true_clf.decision_function(scaler.transform(F[di]))
            actual_frame = D[["family", "cue_kind"]].copy()
            actual_frame["score"] = actual_scores
            actual_auc = family_auc(actual_frame)

            # Exact 252 balanced label assignments across the ten training families.
            families = sorted(tr["family"].unique())
            assert len(families) == 10
            null_aucs = []
            for positive in itertools.combinations(families, 5):
                positive = set(positive)
                shuffled_labels = tr["family"].isin(positive).astype(int).to_numpy()
                assert shuffled_labels.sum() == 25
                null_clf = LogisticRegression(C=cfg["C"], solver="liblinear",
                                              max_iter=2000, random_state=cfg["seed"])
                null_clf.fit(Z, shuffled_labels)
                null_scores = null_clf.decision_function(scaler.transform(F[di]))
                null_frame = D[["family", "cue_kind"]].copy()
                null_frame["score"] = null_scores
                null_auc = family_auc(null_frame)
                null_aucs.append(null_auc)
                all_null.append({"pair": x, "repeat": repeat,
                                 "null_auc": null_auc})
                combinations_done += 1

            null_aucs = np.asarray(null_aucs)
            ge = int(np.sum(null_aucs >= actual_auc - 1e-12))
            p_exact = ge / len(null_aucs)  # exhaustive finite permutation set; no Monte Carlo +1 correction
            summary.append({
                "pair": x, "comparison": y, "repeat": repeat,
                "actual_auc": actual_auc, "null_mean": float(null_aucs.mean()),
                "null_median": float(np.median(null_aucs)),
                "null_q95": float(np.quantile(null_aucs, .95)),
                "null_max": float(null_aucs.max()),
                "null_auc_1_count": int(np.sum(null_aucs == 1.0)),
                "null_auc_0_count": int(np.sum(null_aucs == 0.0)),
                "null_ge_actual": ge, "permutations": len(null_aucs),
                "exact_p": p_exact,
                "actual_above_q95": bool(actual_auc > np.quantile(null_aucs, .95))
            })

            # Exact evaluation-label permutation for the fixed true Probe.
            fam = actual_frame.groupby(["family", "cue_kind"], as_index=False)["score"].mean()
            eval_null, eval_p = exact_eval_label_permutation(fam, actual_auc)
            eval_perm_rows.append({
                "pair": x, "repeat": repeat, "actual_auc": actual_auc,
                "eval_perm_p": eval_p,
                "eval_perm_ge_actual": int(np.sum(eval_null >= actual_auc - 1e-12)),
                "eval_permutations": len(eval_null)
            })

            # Family-swap sensitivity: remove one X family and one Y family at a time.
            xs = fam[fam["cue_kind"] == "x"]["score"].to_numpy()
            ys = fam[fam["cue_kind"] == "y"]["score"].to_numpy()
            assert len(xs) == len(ys) == 5
            subset_aucs = []
            for omit_x in range(5):
                for omit_y in range(5):
                    ax = np.delete(xs, omit_x)
                    ay = np.delete(ys, omit_y)
                    val = float(roc_auc_score([1]*4 + [0]*4, np.r_[ax, ay]))
                    subset_aucs.append(val)
                    family_swap_rows.append({
                        "pair": x, "repeat": repeat,
                        "omitted_x_family": fam[fam["cue_kind"]=="x"].iloc[omit_x]["family"],
                        "omitted_y_family": fam[fam["cue_kind"]=="y"].iloc[omit_y]["family"],
                        "auc": val
                    })
            print(f"{x}/{y} fold {repeat}: actual={actual_auc:.3f}, "
                  f"train-label exact p={p_exact:.4f}, eval-label exact p={eval_p:.4f}")

    exact_summary = pd.DataFrame(summary)
    exact_null = pd.DataFrame(all_null)
    eval_perm = pd.DataFrame(eval_perm_rows)
    family_swap = pd.DataFrame(family_swap_rows)
    assert combinations_done == 20 * 5 * 252 == 25200
    exact_summary.to_csv(ROOT / "exact_training_label_summary.csv", index=False)
    exact_null.to_csv(ROOT / "exact_training_label_null_distribution.csv", index=False)
    eval_perm.to_csv(ROOT / "exact_evaluation_label_permutation.csv", index=False)
    family_swap.to_csv(ROOT / "old_evaluation_family_swap.csv", index=False)
    print("Completed exact training-label fits:", combinations_done)
    return exact_summary, exact_null, eval_perm, family_swap

exact_summary, exact_null, eval_perm, family_swap = exact_training_label_permutations()
display(exact_summary.groupby("pair").agg(
    actual_auc_mean=("actual_auc", "mean"),
    actual_auc_min=("actual_auc", "min"),
    exact_p_max=("exact_p", "max"),
    null_auc1_mean=("null_auc_1_count", "mean"),
    null_q95_mean=("null_q95", "mean")
).round(4))


In [ ]:
# Compare the saved 100-shuffle-per-fit results with the new exhaustive 252-fit null.
saved_shuffle = pd.read_csv(ROOT / "shuffled.csv")
print("Saved random-label training results:", len(saved_shuffle))
print("Exhaustive family-level training-label fits:", len(exact_null))

def describe_auc(values, label):
    s = pd.Series(values)
    return {"distribution": label, "n": len(s), "mean": s.mean(), "median": s.median(),
            "min": s.min(), "q05": s.quantile(.05), "q95": s.quantile(.95),
            "max": s.max(), "auc_0": int((s == 0).sum()), "auc_1": int((s == 1).sum()),
            "auc_le_005": int((s <= .05).sum()), "auc_ge_095": int((s >= .95).sum())}

distribution_summary = pd.DataFrame([
    describe_auc(saved_shuffle["auc"], "Previously saved 100 shuffles per fit"),
    describe_auc(exact_null["null_auc"], "Exhaustive 252 family-label assignments per fit")
])
display(distribution_summary)
distribution_summary.to_csv(ROOT / "shuffle_distribution_summary.csv", index=False)

plt.figure(figsize=(9, 4.5))
plt.hist(saved_shuffle["auc"], bins=np.linspace(0, 1, 21), density=True, alpha=.55,
         label="Previously saved 100 shuffles per fit")
plt.hist(exact_null["null_auc"], bins=np.linspace(0, 1, 21), density=True, alpha=.55,
         label="Exhaustive 252 assignments per fit")
plt.xlabel("Family-level AUC under shuffled training labels")
plt.ylabel("Density")
plt.title("Random-label Probe AUC distributions")
plt.legend()
plt.tight_layout()
plt.savefig(ROOT / "training_label_shuffle_distributions.png", dpi=160)
plt.show()

print("Exact null distribution summary")
display(exact_null["null_auc"].describe(percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]).to_frame("AUC"))
print("AUC=1:", int((exact_null["null_auc"] == 1).sum()))
print("AUC=0:", int((exact_null["null_auc"] == 0).sum()))
print("AUC<=0.05:", int((exact_null["null_auc"] <= .05).sum()))
print("AUC>=0.95:", int((exact_null["null_auc"] >= .95).sum()))
print("Minimum attainable exact permutation p-value:", 1/252)
display(exact_summary.groupby("pair").agg(
    p_min=("exact_p", "min"), p_max=("exact_p", "max"),
    actual_auc_min=("actual_auc", "min"), actual_auc_mean=("actual_auc", "mean"),
    null_auc1_mean=("null_auc_1_count", "mean"), null_q95_mean=("null_q95", "mean")
).round(4))


In [ ]:
# Five new feature families per concept. They are intended to differ from the old training/evaluation descriptions.
NEW_FACTS = json.loads(r"""{
  "banana": [
    [
      "plant_structure",
      "The plant produces new shoots from an underground stem around the base of the parent plant."
    ],
    [
      "leaf_form",
      "Its enormous leaves can split along their veins when exposed to strong winds."
    ],
    [
      "postharvest",
      "The crop is sensitive to cold storage and can develop chilling injury at low temperatures."
    ],
    [
      "ripening",
      "As the fruit ripens, its starch is converted into simpler sugars."
    ],
    [
      "food_use",
      "In many cuisines, the unripe fruit is cooked as a starchy ingredient rather than eaten as a sweet snack."
    ]
  ],
  "apple": [
    [
      "orchard_management",
      "Growers often train the tree into a central-leader shape to distribute sunlight through its branches."
    ],
    [
      "pollination",
      "Insects transfer pollen between blossoms, and many cultivars benefit from cross-pollination."
    ],
    [
      "food_chemistry",
      "Its cut flesh can turn brown when enzymes react with oxygen in the air."
    ],
    [
      "fermentation",
      "Pressed juice can be fermented into a traditional alcoholic drink."
    ],
    [
      "plant_disease",
      "A fungal disease can produce dark, scabby lesions on the leaves and fruit."
    ]
  ],
  "elephant": [
    [
      "social_structure",
      "Adult females commonly remain with their maternal relatives in long-lasting family groups."
    ],
    [
      "reproduction",
      "The pregnancy lasts close to twenty-two months, one of the longest gestation periods among land mammals."
    ],
    [
      "communication",
      "Individuals exchange very low-frequency calls that can travel over long distances."
    ],
    [
      "alloparenting",
      "Young animals are often watched and protected by several adult females in the group."
    ],
    [
      "ecosystem_role",
      "By opening paths and dispersing seeds, the species can substantially reshape its surroundings."
    ]
  ],
  "giraffe": [
    [
      "anatomy",
      "The animal has small skin-covered bony structures on its skull known as ossicones."
    ],
    [
      "feeding_adaptation",
      "A dark, muscular tongue helps it handle leaves among thorny branches."
    ],
    [
      "individual_pattern",
      "The arrangement of markings on its coat differs from one individual to another."
    ],
    [
      "newborn_development",
      "A newborn is usually able to stand and walk within a short time after birth."
    ],
    [
      "circulation",
      "Its cardiovascular system has adaptations that help regulate blood flow to the head."
    ]
  ],
  "piano": [
    [
      "dynamic_control",
      "A performer can produce a broad range of loudness by varying how forcefully a note is played."
    ],
    [
      "tuning",
      "A technician adjusts the tension of many wires using small metal pins set into a heavy frame."
    ],
    [
      "repetition_mechanism",
      "Its action allows a note to be struck again rapidly without the key having to return fully to its resting position."
    ],
    [
      "accompaniment",
      "The instrument is frequently used to accompany singers in art songs and recitals."
    ],
    [
      "ensemble_role",
      "It is often used as a reference when an ensemble tunes before a performance."
    ]
  ],
  "violin": [
    [
      "tuning_system",
      "Its four strings are conventionally tuned at intervals of perfect fifths."
    ],
    [
      "orchestral_organization",
      "In a symphony orchestra, the section is commonly divided into first and second parts."
    ],
    [
      "sound_control",
      "A small mute can be fitted near the bridge to soften the instrument's sound."
    ],
    [
      "chamber_music",
      "It has a central melodic role in the standard string quartet."
    ],
    [
      "intonation",
      "Because its fingerboard has no fixed frets, accurate pitch depends on precise finger placement."
    ]
  ],
  "cat": [
    [
      "scent_marking",
      "The animal can mark familiar objects by rubbing scent glands around its cheeks against them."
    ],
    [
      "vision",
      "Its pupils can change from narrow vertical slits to wide openings as the light level changes."
    ],
    [
      "grooming",
      "Tiny backward-facing structures on its tongue help remove loose hair during grooming."
    ],
    [
      "vocalization",
      "Some individuals greet familiar people with a soft trilling sound."
    ],
    [
      "nutrition",
      "Its metabolism requires nutrients such as taurine that are naturally abundant in animal tissue."
    ]
  ],
  "dog": [
    [
      "early_development",
      "Newborn young have closed eyes and rely heavily on warmth, touch, and scent."
    ],
    [
      "selective_breeding",
      "Different lineages have been selected for specialized work such as herding, tracking, and guarding."
    ],
    [
      "social_signals",
      "Yawning, lip licking, and turning the head away can communicate discomfort or appeasement."
    ],
    [
      "scent_marking",
      "Scent information can be left through glands associated with the animal's paw pads."
    ],
    [
      "reproduction",
      "A group of young born to the same mother at one time is known as a litter."
    ]
  ],
  "lion": [
    [
      "cooperative_hunting",
      "Females in a group may coordinate their efforts when pursuing large prey."
    ],
    [
      "territoriality",
      "Adult males can defend an area against rival coalitions seeking access to the group."
    ],
    [
      "juvenile_coat",
      "Young cubs may have faint spots that become less visible as they mature."
    ],
    [
      "ecological_role",
      "As a top predator, the species can influence the abundance and behavior of other animals in its ecosystem."
    ],
    [
      "social_bonding",
      "Members of a group reinforce relationships through close contact and mutual grooming."
    ]
  ],
  "tiger": [
    [
      "maternal_care",
      "A mother raises her offspring without a permanent family group of adults."
    ],
    [
      "juvenile_period",
      "Young remain dependent on their mother for an extended period before establishing their own range."
    ],
    [
      "territory_marking",
      "Individuals leave scent signals and scratch marks on trees to communicate territorial boundaries."
    ],
    [
      "locomotion",
      "Soft foot pads help the animal move quietly while approaching prey."
    ],
    [
      "habitat_range",
      "Populations occupy very different environments, including mangrove forests and cold northern woodlands."
    ]
  ],
  "dolphin": [
    [
      "nursing",
      "A calf obtains milk from its mother while swimming close beside her."
    ],
    [
      "sleep",
      "The animal can rest one half of its brain while the other half remains alert."
    ],
    [
      "individual_calls",
      "Distinctive signature whistles can help individuals recognize and address one another."
    ],
    [
      "reproduction",
      "Females give birth to live young rather than laying eggs."
    ],
    [
      "birth",
      "A mother may help a newborn reach the water surface for its first breath."
    ]
  ],
  "shark": [
    [
      "electroreception",
      "Specialized sensors around the snout detect weak electrical fields produced by nearby animals."
    ],
    [
      "buoyancy",
      "A large oil-rich liver helps some species regulate their buoyancy."
    ],
    [
      "egg_cases",
      "Some species lay tough egg cases that become attached to rocks or seaweed."
    ],
    [
      "lateral_sensing",
      "A network of canals along the body detects subtle changes in surrounding water pressure."
    ],
    [
      "respiration",
      "Oxygen enters the bloodstream as water passes across the gill surfaces."
    ]
  ],
  "eagle": [
    [
      "nesting",
      "Some species build large nests that are reused and expanded over successive breeding seasons."
    ],
    [
      "sexual_dimorphism",
      "In several species, adult females are noticeably larger than adult males."
    ],
    [
      "chick_development",
      "Newly hatched chicks are covered in down and depend on adults for food and protection."
    ],
    [
      "migration",
      "Some populations travel between separate breeding and wintering regions each year."
    ],
    [
      "keratin_growth",
      "The beak and claws are made largely of keratin and continue to grow throughout life."
    ]
  ],
  "penguin": [
    [
      "egg_incubation",
      "In some species, a parent balances the egg on its feet beneath a warm fold of abdominal skin."
    ],
    [
      "thermal_insulation",
      "Dense waterproof plumage and a layer of body fat help conserve heat in cold water."
    ],
    [
      "recognition",
      "Adults identify their mates and offspring using distinctive calls amid noisy breeding colonies."
    ],
    [
      "countershading",
      "The dark upper surface and pale underside make the bird less conspicuous to predators in the water."
    ],
    [
      "breeding_migration",
      "Some populations travel inland across ice to reach traditional nesting areas."
    ]
  ],
  "butterfly": [
    [
      "migration",
      "Some species travel thousands of kilometres between seasonal habitats."
    ],
    [
      "plant_selection",
      "Taste receptors on the feet help females assess plants before laying eggs."
    ],
    [
      "mimicry",
      "Certain species resemble toxic or unpleasant insects, discouraging predators from attacking them."
    ],
    [
      "adult_lifespan",
      "For many species, the winged adult stage lasts only a relatively short part of the life cycle."
    ],
    [
      "overwintering",
      "Depending on the species, the cold season may be spent as an egg, larva, pupa, or adult."
    ]
  ],
  "spider": [
    [
      "vibration_sensing",
      "Fine sensory hairs on the legs detect vibrations in the surrounding environment."
    ],
    [
      "ballooning",
      "Some species disperse through the air by releasing silk threads that catch moving currents."
    ],
    [
      "venom",
      "Many species use specialized mouthparts to deliver venom when capturing prey."
    ],
    [
      "egg_care",
      "A female may carry or guard a protective egg sac until the young emerge."
    ],
    [
      "silk_use",
      "Silk can be used to build a sheltered retreat, not only a structure for catching prey."
    ]
  ],
  "rose": [
    [
      "propagation",
      "Commercial varieties are often propagated by joining a selected shoot to a hardy root system."
    ],
    [
      "cultivar_selection",
      "Some cultivated varieties have been bred to produce repeated flushes of flowers during one growing season."
    ],
    [
      "grafting",
      "A visible swelling can form where the selected upper variety joins the rootstock."
    ],
    [
      "floriculture",
      "Cut stems are commonly harvested before the flower is fully open to improve vase life."
    ],
    [
      "culinary_use",
      "Petals from suitable varieties can be used to prepare preserves and syrups."
    ]
  ],
  "sunflower": [
    [
      "flower_structure",
      "The large apparent bloom is actually made up of many small individual flowers."
    ],
    [
      "heliotropism",
      "Young developing flower heads can follow the sun across the sky before they mature."
    ],
    [
      "seed_pattern",
      "The seeds are arranged in interlocking spirals that often approximate Fibonacci patterns."
    ],
    [
      "oil_production",
      "The crop is cultivated commercially for oil extracted from its seeds."
    ],
    [
      "phytoremediation",
      "The plant has been studied for its ability to take up certain contaminants from soil."
    ]
  ],
  "carrot": [
    [
      "plant_part",
      "The edible portion is a swollen taproot that stores energy for the plant."
    ],
    [
      "pigment",
      "Orange cultivars contain beta-carotene, which the human body can convert into vitamin A."
    ],
    [
      "taxonomy",
      "The plant belongs to the Apiaceae family, alongside parsley and celery."
    ],
    [
      "cultivar_history",
      "Earlier cultivated forms included purple, white, and yellow varieties."
    ],
    [
      "flowering",
      "When it completes its life cycle, the plant produces flat-topped clusters of small flowers."
    ]
  ],
  "potato": [
    [
      "plant_part",
      "The edible storage organ is a swollen underground stem rather than a true root."
    ],
    [
      "propagation",
      "Growers can start new plants from pieces of the storage organ that contain small buds known as eyes."
    ],
    [
      "food_safety",
      "Green portions may contain elevated levels of glycoalkaloids and should not be eaten."
    ],
    [
      "domestication",
      "The crop was first domesticated in the Andean region of South America."
    ],
    [
      "fruiting",
      "After flowering, the plant can produce small berry-like fruits above the soil."
    ]
  ],
  "lemon": [
    [
      "acidity",
      "Its juice is strongly acidic because it contains a high concentration of citric acid."
    ],
    [
      "peel_oils",
      "Small oil glands in the rind release aromatic compounds when the surface is pressed."
    ],
    [
      "phenology",
      "In warm climates, a tree can carry blossoms and mature fruit at the same time."
    ],
    [
      "preservation",
      "A traditional preserved form is cured with salt and used as a condiment."
    ],
    [
      "culinary_chemistry",
      "Its juice is used in marinades because acidity changes the structure of proteins in food."
    ]
  ],
  "orange": [
    [
      "fruit_structure",
      "The fruit is divided into segments separated by thin membranes."
    ],
    [
      "confectionery",
      "The peel can be candied and used to flavor sweets and baked goods."
    ],
    [
      "floral_product",
      "The blossoms can be distilled or processed into a fragrant water used in cooking."
    ],
    [
      "orchard_management",
      "Commercial growers often use grafted trees to keep fruit quality consistent."
    ],
    [
      "botany",
      "In botanical classification, the fruit belongs to a type known as a hesperidium."
    ]
  ],
  "bread": [
    [
      "fermentation",
      "Yeast releases carbon dioxide during fermentation, causing the dough to expand."
    ],
    [
      "gluten",
      "Proteins in the flour form an elastic network when the ingredients are mixed and worked."
    ],
    [
      "baking",
      "The interior structure becomes set as starch gelatinizes in the heat of the oven."
    ],
    [
      "starter_culture",
      "A fermented starter can be maintained and refreshed over many days."
    ],
    [
      "staling",
      "The interior becomes firm over time as starch molecules reorganize after baking."
    ]
  ],
  "rice": [
    [
      "crop_type",
      "The grain is the seed of a grass species cultivated in fields that are often flooded."
    ],
    [
      "milling",
      "Polishing removes the bran and germ to produce the familiar white grain."
    ],
    [
      "starch",
      "Some cultivars become especially sticky because they contain a high proportion of amylopectin."
    ],
    [
      "cultivation",
      "Farmers in many regions transplant young seedlings from nurseries into prepared fields."
    ],
    [
      "processing",
      "Milling first removes the inedible outer husk from the harvested grain."
    ]
  ],
  "bicycle": [
    [
      "power_transfer",
      "The rider transfers energy to the rear wheel through a chain driven by pedals."
    ],
    [
      "gearing",
      "Changing the relative sizes of sprockets alters the gear ratio and the effort required to move."
    ],
    [
      "frame",
      "Many frames are built from joined tubes arranged in a roughly diamond-shaped structure."
    ],
    [
      "balance",
      "The rider maintains stability through steering corrections and small shifts in body position."
    ],
    [
      "derailleur",
      "A movable mechanism guides the chain between sprockets to change the gear."
    ]
  ],
  "motorcycle": [
    [
      "transmission",
      "Engine output commonly passes through a clutch and gearbox before reaching the rear wheel."
    ],
    [
      "throttle",
      "Acceleration is controlled by rotating a hand grip."
    ],
    [
      "engine_measurement",
      "Many models describe engine size by displacement measured in cubic centimetres."
    ],
    [
      "gear_selection",
      "The rider commonly selects gears using a lever operated by the foot."
    ],
    [
      "final_drive",
      "Some models transmit power to the rear wheel through a shaft rather than a chain."
    ]
  ],
  "train": [
    [
      "coupling",
      "Several rail vehicles can be connected together to form one moving consist."
    ],
    [
      "power_supply",
      "Electric services may draw power from overhead wires or a conductor rail beside the track."
    ],
    [
      "wheel_design",
      "A flange on each wheel helps guide the vehicle along the rail."
    ],
    [
      "traffic_control",
      "Signals and interlocking systems coordinate movements and prevent conflicting routes."
    ],
    [
      "service_pattern",
      "Passenger services typically stop at fixed stations according to a published timetable."
    ]
  ],
  "airplane": [
    [
      "aerodynamics",
      "Air flowing around the wings produces the lift needed to support the aircraft in flight."
    ],
    [
      "flight_controls",
      "Movable surfaces adjust pitch, roll, and yaw while the aircraft is airborne."
    ],
    [
      "cabin_environment",
      "The passenger cabin is pressurized to maintain breathable conditions at cruising altitude."
    ],
    [
      "propulsion",
      "Turbofan engines create thrust by accelerating a large mass of air backward."
    ],
    [
      "navigation",
      "Pilots combine radio navigation with onboard instruments to follow a planned route."
    ]
  ],
  "boat": [
    [
      "buoyancy",
      "Its hull displaces enough water to support the vessel's weight."
    ],
    [
      "propulsion",
      "Oars, sails, or a propeller can provide the force needed to move it forward."
    ],
    [
      "navigation",
      "Charts, buoys, and coastal landmarks can help determine its position and route."
    ],
    [
      "operating_position",
      "The design is intended to keep the vessel at the water surface during normal operation."
    ],
    [
      "stability",
      "A keel can improve directional stability and reduce sideways drift."
    ]
  ],
  "submarine": [
    [
      "pressure_resistance",
      "Its pressure hull is engineered to withstand the force of surrounding water at depth."
    ],
    [
      "sonar",
      "Sound waves are transmitted and received to detect and locate objects underwater."
    ],
    [
      "life_support",
      "The crew works inside a sealed compartment with carefully controlled air conditions."
    ],
    [
      "underwater_endurance",
      "The vessel can remain below the surface for extended periods without returning to the air."
    ],
    [
      "navigation",
      "Inertial measurements help estimate its position when external navigation signals are unavailable."
    ]
  ],
  "hammer": [
    [
      "striking_face",
      "The striking face may be smooth or textured depending on the material being worked."
    ],
    [
      "impact_energy",
      "The energy delivered in a strike depends on the head's mass and the speed of the swing."
    ],
    [
      "materials",
      "A hardened steel head is commonly attached to a wooden or composite handle."
    ],
    [
      "carpentry",
      "A carpenter may use it to tap joints together or set another cutting tool into position."
    ],
    [
      "soft_face",
      "Some versions use a replaceable soft face to avoid damaging delicate surfaces."
    ]
  ],
  "screwdriver": [
    [
      "interchangeable_bits",
      "Interchangeable bits allow one handle to work with several fastener-head patterns."
    ],
    [
      "electrical_safety",
      "An insulated handle is used for certain electrical maintenance tasks."
    ],
    [
      "ratchet",
      "A ratcheting mechanism permits repeated turns without lifting and repositioning the handle."
    ],
    [
      "precision_work",
      "Small precision versions are used to adjust fasteners inside electronic devices."
    ],
    [
      "torque_control",
      "Some powered versions include a clutch that limits torque and helps prevent over-tightening."
    ]
  ],
  "scissors": [
    [
      "spring_action",
      "Spring-loaded versions reopen automatically after each squeeze."
    ],
    [
      "tailoring",
      "Long-bladed versions are designed to cut fabric in continuous strokes."
    ],
    [
      "pivot_adjustment",
      "The central screw can be adjusted to change how tightly the two blades meet."
    ],
    [
      "medical_design",
      "Some specialized designs have blunt tips for use around bandages."
    ],
    [
      "maintenance",
      "The cutting performance can depend on the alignment and tension of the pivot."
    ]
  ],
  "knife": [
    [
      "culinary_technique",
      "A chef may use a rocking motion to mince herbs on a cutting board."
    ],
    [
      "edge_geometry",
      "Some blades are sharpened on only one side for specialized cutting tasks."
    ],
    [
      "maintenance",
      "The cutting edge can be maintained by honing it against a steel rod."
    ],
    [
      "blade_profiles",
      "Different blade profiles are designed for slicing, paring, or chopping."
    ],
    [
      "manufacture",
      "Some high-end blades are forged from layers of steel that form visible patterns."
    ]
  ],
  "clock": [
    [
      "quartz_timing",
      "A quartz oscillator vibrates at a stable frequency to regulate timekeeping."
    ],
    [
      "mechanical_power",
      "Mechanical models store energy in a wound spring or a suspended weight."
    ],
    [
      "escapement",
      "An escapement releases stored energy in controlled increments."
    ],
    [
      "radio_sync",
      "Some devices synchronize with radio signals broadcast by national time services."
    ],
    [
      "atomic_standard",
      "Highly precise time standards rely on stable transitions in atoms."
    ]
  ],
  "calendar": [
    [
      "leap_year",
      "The Gregorian system adds an extra day in most years that are divisible by four."
    ],
    [
      "century_rule",
      "Century years follow a special exception unless they are also divisible by four hundred."
    ],
    [
      "lunar_systems",
      "Some systems organize months according to the changing phases of the moon."
    ],
    [
      "fiscal_year",
      "A fiscal year used by an organization may begin in a month other than January."
    ],
    [
      "weekly_cycle",
      "The conventional seven-day cycle continues independently of the length of a month."
    ]
  ],
  "refrigerator": [
    [
      "refrigeration_cycle",
      "A vapour-compression cycle transfers heat from an insulated cabinet into the surrounding room."
    ],
    [
      "thermostat",
      "A thermostat switches the compressor on and off to maintain a chosen temperature."
    ],
    [
      "defrost",
      "Frost-free designs periodically warm a cooling coil to remove accumulated ice."
    ],
    [
      "door_seal",
      "A flexible door seal helps prevent warm air from entering the cooled compartment."
    ],
    [
      "freezer_compartment",
      "Some models include a separate compartment maintained below the freezing point of water."
    ]
  ],
  "oven": [
    [
      "temperature_control",
      "A thermostat regulates heating elements or a gas burner to maintain a selected temperature."
    ],
    [
      "convection",
      "Some models circulate hot air with an internal fan to distribute heat more evenly."
    ],
    [
      "broiling",
      "A broiler cooks food using intense radiant heat from an element positioned above it."
    ],
    [
      "self_cleaning",
      "A self-cleaning cycle heats the interior until food residue is reduced to ash."
    ],
    [
      "calibration",
      "The appliance may need calibration if its actual temperature differs from the selected setting."
    ]
  ],
  "camera": [
    [
      "exposure",
      "Exposure depends on shutter duration, aperture size, and the sensitivity of the recording surface."
    ],
    [
      "autofocus",
      "An autofocus system estimates subject distance and adjusts the position of a lens element."
    ],
    [
      "metadata",
      "Digital image files can store information about exposure settings and the lens used."
    ],
    [
      "long_exposure",
      "A long exposure can turn moving points of light into continuous streaks."
    ],
    [
      "stabilization",
      "Optical stabilization compensates for small movements made while holding the device."
    ]
  ],
  "telescope": [
    [
      "tracking_mount",
      "A tracking mount compensates for Earth's rotation to keep a celestial target centered."
    ],
    [
      "optical_design",
      "Reflecting designs use a curved primary mirror to gather and focus incoming light."
    ],
    [
      "finder",
      "A small finder scope helps align the main instrument with a distant target."
    ],
    [
      "aperture",
      "A larger aperture collects more light from faint objects in the night sky."
    ],
    [
      "spectroscopy",
      "Spectroscopy can reveal the chemical composition of distant stars."
    ]
  ]
}""")
NEW_FRAMES = [
    "An additional fact is that {s}",
    "One identifying detail is this: {s}",
    "The following characteristic applies: {s}",
    "In this context, it is relevant that {s}",
]
NEUTRAL_INSTRUCTION = "Read the context carefully."

new_rows = []
for concept, items in NEW_FACTS.items():
    assert len(items) == 5, (concept, len(items))
    for family_index, (domain, sentence) in enumerate(items):
        for variant, frame in enumerate(NEW_FRAMES):
            wrapped = frame.format(s=sentence)
            prompt = f"Instruction: {NEUTRAL_INSTRUCTION}\n\nContext:\n{wrapped}\n\nGive one short response about the context."
            new_rows.append({
                "concept": concept,
                "family": f"{concept}_independent_{family_index}",
                "family_index": family_index,
                "domain": domain,
                "variant": variant,
                "text": sentence,
                "wrapped_text": wrapped,
                "prompt": prompt
            })
new_eval = pd.DataFrame(new_rows)
assert len(new_eval) == 40 * 5 * 4 == 800
assert new_eval["prompt"].nunique() == 800

# Check exact text leakage against original train/evaluation descriptions.
def norm(s):
    return re.sub(r"\s+", " ", str(s).casefold()).strip()
old_texts = set(train["text"].map(norm))
old_eval_texts = set(design.loc[design["cue_kind"].isin(["x", "y"]), "text"].map(norm))
assert not set(new_eval["text"].map(norm)) & old_texts
assert not set(new_eval["text"].map(norm)) & old_eval_texts
new_eval.to_csv(ROOT / "independent_eval_cues.csv", index=False)
print("Prepared independent candidate set:", len(new_eval), "prompts /",
      new_eval["family"].nunique(), "concept-family descriptions")
display(new_eval[["concept", "family_index", "domain", "text"]].drop_duplicates().head(20))


In [ ]:
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."
torch.set_num_threads(4)
torch.manual_seed(cfg["seed"])
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

model_name = cfg["model"]
revision = cfg["revision"]
tokenizer = AutoTokenizer.from_pretrained(
    model_name, revision=revision, use_fast=True, trust_remote_code=False
)
assert tokenizer.is_fast
model = AutoModelForCausalLM.from_pretrained(
    model_name, revision=revision, dtype=torch.float16,
    attn_implementation="sdpa", device_map={"": "cuda:0"},
    trust_remote_code=False
).eval()
assert model.config.num_hidden_layers == 32
assert model.config.hidden_size == 3072

new_prompts = new_eval["prompt"].tolist()
new_features = []
start = time.perf_counter()
with torch.inference_mode():
    for i, prompt in enumerate(new_prompts):
        rendered = tokenizer.apply_chat_template(
            [{"role": "system", "content": cfg["system"]},
             {"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True
        )
        batch = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
        assistant_start = batch["input_ids"].shape[1] - 1
        result = model(**batch.to("cuda"), output_hidden_states=True,
                       use_cache=False, return_dict=True)
        vector = result.hidden_states[31][0, assistant_start].float().cpu().numpy()
        new_features.append(vector)
        del result
        if (i + 1) % 100 == 0:
            print(f"Extracted {i+1}/{len(new_prompts)}")
    new_features = np.stack(new_features).astype(np.float32)

np.savez_compressed(ROOT / "independent_eval_features.npz", features=new_features)
new_eval.to_csv(ROOT / "independent_eval_index.csv", index=False)
print("Hidden-state extraction complete:", new_features.shape,
      "seconds:", round(time.perf_counter() - start, 1))
del model
torch.cuda.empty_cache()
gc.collect()


In [ ]:
# Score the new independent descriptions with the same training data, preprocessing, and Probe family.
new_F = np.load(ROOT / "independent_eval_features.npz")["features"].astype(np.float64)
assert new_F.shape == (800, 3072)
new_eval = pd.read_csv(ROOT / "independent_eval_index.csv")

def exact_family_permutation_p(frame, observed):
    scores = frame["score"].to_numpy()
    pvals = []
    for positive in itertools.combinations(range(10), 5):
        labels = np.zeros(10, dtype=int)
        labels[list(positive)] = 1
        pvals.append(float(roc_auc_score(labels, scores)))
    pvals = np.asarray(pvals)
    return int(np.sum(pvals >= observed - 1e-12)) / 252, pvals

new_results, new_family_scores, new_leaveout = [], [], []
for x, y in cfg["pairs"].items():
    for repeat in range(cfg["fit_repeats"]):
        tr = train[train["concept"].isin([x, y]) & (train["family_index"] != repeat)].copy()
        ti = np.array([lookup[p] for p in tr["prompt"]], dtype=int)
        X = F[ti]
        scaler = StandardScaler().fit(X)
        Z = scaler.transform(X)
        labels = (tr["concept"] == x).astype(int).to_numpy()
        clf = LogisticRegression(C=cfg["C"], solver="liblinear",
                                 max_iter=2000, random_state=cfg["seed"])
        clf.fit(Z, labels)

        # Old evaluation AUC for direct comparison.
        old_D = design[(design["pair"] == x) &
                       (design["condition"] == "neutral") &
                       (design["cue_kind"].isin(["x", "y"]))].copy()
        old_idx = np.array([lookup[p] for p in old_D["prompt"]], dtype=int)
        old_D["score"] = clf.decision_function(scaler.transform(F[old_idx]))
        old_fam = old_D.groupby(["family", "cue_kind"], as_index=False)["score"].mean()
        old_auc = float(roc_auc_score((old_fam["cue_kind"] == "x").astype(int),
                                      old_fam["score"]))

        # New, unseen description families.
        nd = new_eval[new_eval["concept"].isin([x, y])].copy()
        row_idx = nd.index.to_numpy()
        nd["score"] = clf.decision_function(scaler.transform(new_F[row_idx]))
        nd["cue_kind"] = np.where(nd["concept"] == x, "x", "y")
        fam = nd.groupby(["family", "cue_kind"], as_index=False)["score"].mean()
        assert len(fam) == 10 and (fam["cue_kind"] == "x").sum() == 5
        new_auc = float(roc_auc_score((fam["cue_kind"] == "x").astype(int),
                                      fam["score"]))
        p_exact, perm = exact_family_permutation_p(fam, new_auc)

        xs = fam[fam["cue_kind"] == "x"].reset_index(drop=True)
        ys = fam[fam["cue_kind"] == "y"].reset_index(drop=True)
        subset = []
        for omit_x in range(5):
            for omit_y in range(5):
                ax = xs.drop(index=omit_x)["score"].to_numpy()
                ay = ys.drop(index=omit_y)["score"].to_numpy()
                val = float(roc_auc_score([1]*4 + [0]*4, np.r_[ax, ay]))
                subset.append(val)
                new_leaveout.append({
                    "pair": x, "repeat": repeat,
                    "omitted_x_family": xs.loc[omit_x, "family"],
                    "omitted_y_family": ys.loc[omit_y, "family"],
                    "auc": val
                })

        for row in fam.to_dict("records"):
            new_family_scores.append({"pair": x, "comparison": y,
                                       "repeat": repeat, **row})
        new_results.append({
            "pair": x, "comparison": y, "repeat": repeat,
            "old_eval_auc": old_auc, "new_independent_auc": new_auc,
            "delta_new_minus_old": new_auc - old_auc,
            "new_exact_eval_p": p_exact,
            "new_leaveout_min": min(subset),
            "new_leaveout_mean": float(np.mean(subset)),
            "new_leaveout_max": max(subset),
            "new_leaveout_splits_ge_09": int(np.sum(np.asarray(subset) >= .9)),
            "new_leaveout_splits": 25
        })
        print(f"{x}/{y} fold {repeat}: old={old_auc:.3f}, new={new_auc:.3f}, "
              f"exact eval p={p_exact:.4f}, leaveout min={min(subset):.3f}")

new_results = pd.DataFrame(new_results)
new_family_scores = pd.DataFrame(new_family_scores)
new_leaveout = pd.DataFrame(new_leaveout)
new_results.to_csv(ROOT / "independent_eval_probe_summary.csv", index=False)
new_family_scores.to_csv(ROOT / "independent_eval_family_scores.csv", index=False)
new_leaveout.to_csv(ROOT / "independent_eval_family_leaveout.csv", index=False)

display(new_results.groupby("pair").agg(
    old_auc_mean=("old_eval_auc", "mean"),
    new_auc_mean=("new_independent_auc", "mean"),
    new_auc_min=("new_independent_auc", "min"),
    new_exact_p_max=("new_exact_eval_p", "max"),
    leaveout_min=("new_leaveout_min", "min")
).round(4))


## 出力と解釈上の注意

結果は `/content/probe_recheck/` に保存されます。最後のセルで、主要なCSVと図をZIPにまとめてダウンロードします。

- `exact_training_label_summary.csv`: 100個の学習構成ごとの実測AUC・252通りの学習ラベル置換結果。
- `exact_training_label_null_distribution.csv`: 25,200個すべてのシャッフルAUC。
- `exact_evaluation_label_permutation.csv`: 固定Probeに対する評価ラベル置換検定。
- `old_evaluation_family_swap.csv`: 既存評価ファミリーの除外感度。
- `independent_eval_cues.csv`: 新しい説明文200件と4種類のプロンプト枠。
- `independent_eval_probe_summary.csv`: 新しい説明文でのProbe評価。
- `independent_eval_family_scores.csv` / `independent_eval_family_leaveout.csv`: ファミリー単位の詳細。

### 統計的な注意
- 学習ラベル置換検定の最小p値は1/252です。20ペアを同時に検定する場合、単純なBonferroni補正では最小値でも0.0794となるため、個々の未補正p値を確証的な結論として扱わないでください。
- 5つの学習構成は重複した学習ファミリーを使う感度分析であり、独立な反復ではありません。
- 評価ラベル置換検定は、固定したProbeのスコアと評価ラベルの対応を検定します。学習ラベル置換検定の代わりにはなりません。
- 新しい評価文は独立した説明ファミリーを意図していますが、AI作成文の意味的重複・曖昧さは残り得ます。人手レビュー前の結果は探索的に扱ってください。
- 新しい説明文でAUCが高くても、概念一般への汎化や抑制条件の因果効果まで証明するものではありません。
- `hidden_states[31]` と `assistant_start` を使い、元の実験と同じ抽出位置に揃えます。回答生成は行いません。


In [ ]:
# Bundle the analysis outputs for review in ChatGPT.
import zipfile
from google.colab import files

result_zip = ROOT / "probe_recheck_results.zip"
include = [
    "exact_training_label_summary.csv",
    "exact_training_label_null_distribution.csv",
    "shuffle_distribution_summary.csv",
    "training_label_shuffle_distributions.png",
    "exact_evaluation_label_permutation.csv",
    "old_evaluation_family_swap.csv",
    "independent_eval_cues.csv",
    "independent_eval_probe_summary.csv",
    "independent_eval_family_scores.csv",
    "independent_eval_family_leaveout.csv",
    "exact_training_shuffle_histogram.png",
]
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in include:
        path = ROOT / name
        if path.exists():
            z.write(path, arcname=name)
print("Created results bundle:", result_zip, result_zip.stat().st_size, "bytes")
files.download(str(result_zip))


In [ ]:
# ============================================================
# Save experiment results to GitHub
# ============================================================

from google.colab import userdata
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import base64
import hashlib
import json
import requests

# Configuration
REPO = "ryouy/GraduationThesis"
BRANCH = "main"
RESULT_ROOT = Path("/content/probe_recheck")

# Read GitHub token from Colab Secrets
# Create a secret named GITHUB_TOKEN
TOKEN = userdata.get("GITHUB_TOKEN")

if not TOKEN:
    raise ValueError("GITHUB_TOKEN is not configured in Colab Secrets.")

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

API = f"https://api.github.com/repos/{REPO}/contents"

# Create a timestamped results directory
timestamp = datetime.now(
    ZoneInfo("Asia/Tokyo")
).strftime("%Y%m%d_%H%M%S")

DEST = f"1110_probe_reanalysis/results/{timestamp}"

# Files to save
FILES = [
    "exact_training_label_summary.csv",
    "exact_training_label_null_distribution.csv",
    "exact_evaluation_label_permutation.csv",
    "old_evaluation_family_swap.csv",
    "shuffle_distribution_summary.csv",
    "training_label_shuffle_distributions.png",
    "independent_eval_cues.csv",
    "independent_eval_index.csv",
    "independent_eval_features.npz",
    "independent_eval_probe_summary.csv",
    "independent_eval_family_scores.csv",
    "independent_eval_family_leaveout.csv",
]

def upload_to_github(local_path, github_path):
    data = local_path.read_bytes()

    payload = {
        "message": f"Save Probe reanalysis: {local_path.name}",
        "content": base64.b64encode(data).decode("ascii"),
        "branch": BRANCH,
    }

    url = f"{API}/{github_path}"

    response = requests.put(
        url,
        headers=HEADERS,
        json=payload,
        timeout=180,
    )
    response.raise_for_status()

    return hashlib.sha256(data).hexdigest()

# Upload results
saved = {}
missing = []

for filename in FILES:
    local_path = RESULT_ROOT / filename

    if not local_path.exists():
        print(f"SKIP (not found): {filename}")
        missing.append(filename)
        continue

    github_path = f"{DEST}/{filename}"

    try:
        checksum = upload_to_github(local_path, github_path)
        saved[filename] = {
            "sha256": checksum,
            "bytes": local_path.stat().st_size,
        }
        print(f"OK: {filename}")

    except Exception as e:
        print(f"ERROR: {filename}: {e}")
        raise

# Save manifest
manifest = {
    "experiment": "Probe reanalysis and independent generalization",
    "timestamp_jst": timestamp,
    "repository": REPO,
    "result_directory": DEST,
    "saved_files": saved,
    "missing_files": missing,
}

manifest_path = RESULT_ROOT / "github_result_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

upload_to_github(
    manifest_path,
    f"{DEST}/github_result_manifest.json",
)

print("\nGitHub save completed.")
print(f"https://github.com/{REPO}/tree/{BRANCH}/{DEST}")